In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
%run ./UDF/udf_silver_incremental_ingest

In [0]:


src_bronze_path = "/Volumes/data_governance/bronze_cost_monitoring/billing_usage_volume"
tgt_silver_table = "data_governance.silver_cost_monitoring.billing_usage"



In [0]:
df=silver_incremental_ingest(src_bronze_path,tgt_silver_table)

In [0]:
if df.count()==0:
    dbutils.notebook.exit("No new records to load")
else:
    pass

In [0]:
df = df.withColumns({
    "usage_start_time": to_timestamp("usage_start_time"),
    "usage_end_time": to_timestamp("usage_end_time"),
    "usage_date": to_date("usage_date"),
    "load_timestamp": to_timestamp("load_timestamp"),
    # Calculate duration in minutes
    "usage_duration_minutes": (unix_timestamp("usage_end_time") - unix_timestamp("usage_start_time")) / 60
})

In [0]:
df= df.select(
    # Keep all existing base columns (account_id, workspace_id, sku_name, etc.)
    "*",
    
    # 1. Identity Metadata Flattening
    col("identity_metadata.run_as").alias("user_principal"),
    col("identity_metadata.created_by").alias("created_by"),
    col("identity_metadata.owned_by").alias("resource_owner"),
    
    # 2. Product Features Flattening
    col("product_features.is_serverless").cast("boolean").alias("is_serverless"),
    col("product_features.is_photon").cast("boolean").alias("is_photon"),
    col("product_features.sql_tier").alias("sql_tier"),
    col("product_features.jobs_tier").alias("jobs_tier"),
    col("product_features.serving_type").alias("serving_type"),
    col("product_features.networking.connectivity_type").alias("connectivity_type"),
    
    # 3. Usage Metadata Flattening (Key for Joins)
    col("usage_metadata.cluster_id").alias("cluster_id"),
    col("usage_metadata.job_id").alias("job_id"),
    col("usage_metadata.warehouse_id").alias("warehouse_id"),
    col("usage_metadata.node_type").alias("node_type"),
    col("usage_metadata.notebook_path").alias("notebook_path"),
    col("usage_metadata.source_region").alias("source_region"),
    col("usage_metadata.destination_region").alias("destination_region"),
    col("usage_metadata.budget_policy_id").alias("budget_policy_id"),
    
    # Special: Full Table Path for Lineage Joins
    concat_ws(".", 
                col("usage_metadata.uc_table_catalog"),
                col("usage_metadata.uc_table_schema"), 
                col("usage_metadata.uc_table_name")
               ).alias("full_table_path")
).drop("identity_metadata", "product_features", "usage_metadata") # Remove originals

# Show the results to verify
display(df)

In [0]:
df = df.withColumns({
    # Fill null workspace_id for account-level records
    "workspace_id": coalesce(col("workspace_id"), lit("ACCOUNT_LEVEL")),
    # Standardize SKU to uppercase for consistent joining
    "sku_name": upper(col("sku_name")),
    # Ensure quantity is a high-precision decimal
    "usage_quantity": col("usage_quantity").cast("decimal(18,6)")
}).filter(col("record_type") == "ORIGINAL") # Removing adjustment records for simple reporting

# Deduplicate based on record_id
df = df.dropDuplicates(["record_id"])

In [0]:


df = df.withColumns({
    # 1. High-precision quantity for financial accuracy
    "usage_quantity": col("usage_quantity").cast("decimal(38,10)"),
    
    # 2. Unified Resource ID for easy joining with Audit/Lineage
    "resource_id": coalesce(
        col("cluster_id"), 
        col("job_id"), 
        col("warehouse_id"), 
        lit("ACCOUNT_LEVEL")
    ),
    
    # 3. Clean Categorization from SKU Name
    "service_family": when(col("sku_name").contains("COMPUTE"), "Compute")
                       .when(col("sku_name").contains("STORAGE"), "Storage")
                       .when(col("sku_name").contains("EGRESS") | col("sku_name").contains("CONNECTIVITY"), "Networking")
                       .otherwise("Other"),
    
    # 4. Standardize Workspace identity
    "workspace_id": when(col("workspace_id") == "ACCOUNT_LEVEL", "0") # Standardize to a numeric string
                     .otherwise(col("workspace_id")),
                     
    # 5. Null handling for Strings (removes literal "null" text)
    "user_principal": when(col("user_principal") == "null", lit(None)).otherwise(col("user_principal"))
})

# Final Drop of columns that are no longer needed after these refinements
df = df.drop("custom_tags")

In [0]:
df.limit(20).display()

In [0]:
# Writing with Liquid Clustering
df.write\
 .format("delta")\
 .mode("append") \
 .partitionBy("billing_origin_product") \
 .saveAsTable(tgt_silver_table)

In [0]:
%sql
select * from data_governance.silver_cost_monitoring.billing_usage;